# US500 Data Fetcher & Feature Engineering
Downloads historical data, engineers all features, validates trading rules,
and produces exploratory charts.

**Run in Google Colab (GPU) or locally.** All files save to local disk.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q yfinance pandas numpy scikit-learn xgboost lightgbm shap smartmoneyconcepts pandas-ta requests seaborn matplotlib

In [ ]:
# Cell 2 — Imports & paths
import os, sys, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')
sns.set_palette('bright')

BASE_PATH = "/content" if "google.colab" in sys.modules else os.path.abspath("..")
DATA_RAW = os.path.join(BASE_PATH, "data", "raw")
DATA_FEAT = os.path.join(BASE_PATH, "data", "features")
os.makedirs(DATA_RAW, exist_ok=True)
os.makedirs(DATA_FEAT, exist_ok=True)

TICKERS = {
    "vix":   "^VIX",
    "dxy":   "DX-Y.NYB",
    "us10y": "^TNX",
    "oil":   "CL=F",
    "rut":   "^RUT",
    "sp500": "^GSPC",
}

print(f"Base path: {BASE_PATH}")
print(f"Raw data:  {DATA_RAW}")
print(f"Features:  {DATA_FEAT}")

In [ ]:
# Cell 3 — Retry fetch utility
def fetch_with_retry(ticker: str, period: str, interval: str, retries: int = 3) -> pd.DataFrame:
    """Download yfinance data with retries on failure."""
    for attempt in range(1, retries + 1):
        try:
            df = yf.download(ticker, period=period, interval=interval, progress=False)
            if df is not None and not df.empty:
                print(f"  ✅ {ticker} ({interval}) — {len(df)} rows")
                return df
        except Exception as e:
            print(f"  ⚠️ Attempt {attempt} for {ticker} failed: {e}")
            time.sleep(2)
    print(f"  ❌ {ticker} — all retries failed")
    return pd.DataFrame()

In [ ]:
# Cell 4 — Download 5yr daily data, save per-ticker parquet
print("Downloading 5-year DAILY data...")
daily_frames = {}
for name, ticker in TICKERS.items():
    df = fetch_with_retry(ticker, period="5y", interval="1d")
    if not df.empty:
        path = os.path.join(DATA_RAW, f"{name}_daily.parquet")
        df.to_parquet(path)
        daily_frames[name] = df
print(f"\nDownloaded {len(daily_frames)} / {len(TICKERS)} tickers")

In [ ]:
# Cell 5 — Download 2yr hourly data
print("Downloading 2-year HOURLY data...")
hourly_frames = {}
for name, ticker in TICKERS.items():
    df = fetch_with_retry(ticker, period="2y", interval="1h")
    if not df.empty:
        path = os.path.join(DATA_RAW, f"{name}_hourly.parquet")
        df.to_parquet(path)
        hourly_frames[name] = df
print(f"\nDownloaded {len(hourly_frames)} / {len(TICKERS)} hourly tickers")

In [ ]:
# Cell 6 — Download 2yr 15-min US500 only (for M15 order-flow analysis)
print("Downloading 2-year M15 SP500 data...")
sp500_m15 = fetch_with_retry("^GSPC", period="60d", interval="15m")
if not sp500_m15.empty:
    sp500_m15.to_parquet(os.path.join(DATA_RAW, "sp500_m15.parquet"))
    print(f"M15 data saved: {len(sp500_m15)} bars")
else:
    print("M15 download failed — yfinance limits 15m to ~60 days")

In [ ]:
# Cell 7 — Merge all daily into one aligned DataFrame
merged = pd.DataFrame()
for name, df in daily_frames.items():
    cols = df.columns
    if isinstance(cols, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    renamed = df[["Open", "High", "Low", "Close", "Volume"]].copy()
    renamed.columns = [f"{name}_{c.lower()}" for c in renamed.columns]
    if merged.empty:
        merged = renamed
    else:
        merged = merged.join(renamed, how="outer")

merged.dropna(subset=["sp500_close"], inplace=True)
merged.sort_index(inplace=True)
print(f"Merged daily DataFrame: {merged.shape}")
merged.tail(3)

In [ ]:
# Cell 8 — Feature engineering (all features)
df = merged.copy()

# --- Price features ---
df["sp500_return"] = df["sp500_close"].pct_change() * 100
df["sp500_next_return"] = df["sp500_return"].shift(-1)
df["sp500_direction"] = np.where(df["sp500_next_return"] > 0, 1, -1)

# --- VIX features ---
vix = df["vix_close"]
df["vix_pct"] = vix.pct_change() * 100
df["vix_ema5"] = vix.ewm(span=5).mean()
df["vix_ema20"] = vix.ewm(span=20).mean()
df["vix_trend"] = np.where(df["vix_ema5"] > df["vix_ema20"], 1, -1)
df["vix_spike_20"] = (vix > 20).astype(int)
df["vix_spike_25"] = (vix > 25).astype(int)
df["vix_spike_30"] = (vix > 30).astype(int)
df["vix_3day_change"] = vix.pct_change(3) * 100

# --- Macro features ---
df["dxy_pct"] = df["dxy_close"].pct_change() * 100
df["dxy_3day"] = df["dxy_close"].pct_change(3) * 100
df["us10y_pct"] = df["us10y_close"].pct_change() * 100
df["us10y_3day"] = df["us10y_close"].pct_change(3) * 100
df["oil_pct"] = df["oil_close"].pct_change() * 100
df["oil_3day"] = df["oil_close"].pct_change(3) * 100

rut_ret = df["rut_close"].pct_change() * 100
df["rut_vs_sp500"] = rut_ret - df["sp500_return"]
df["rut_leading"] = np.where(df["rut_vs_sp500"] > 0, 1, -1)

# --- Time features ---
df["day_of_week"] = pd.to_datetime(df.index).dayofweek
df["is_monday"] = (df["day_of_week"] == 0).astype(int)
df["is_friday"] = (df["day_of_week"] == 4).astype(int)
df["is_tuesday"] = (df["day_of_week"] == 1).astype(int)
df["week_of_month"] = pd.to_datetime(df.index).day.map(lambda d: min((d - 1) // 7 + 1, 4))

# --- Regime features ---
df["overnight_gap"] = ((df["sp500_open"] - df["sp500_close"].shift(1)) / df["sp500_close"].shift(1)) * 100
df["prev_day_return"] = df["sp500_return"].shift(1)
up = (df["sp500_return"] > 0).astype(int)
down = (df["sp500_return"] < 0).astype(int)
df["consecutive_up"] = up.groupby((up != up.shift()).cumsum()).cumsum()
df["consecutive_down"] = down.groupby((down != down.shift()).cumsum()).cumsum()
df["volatility_5d"] = df["sp500_return"].rolling(5).std()

# --- Interaction features ---
df["vix_up_dxy_up"] = ((df["vix_pct"] > 1) & (df["dxy_pct"] > 0.2)).astype(int)
df["vix_down_rut_up"] = ((df["vix_pct"] < -1) & (df["rut_vs_sp500"] > 0)).astype(int)
df["vix_spike_friday"] = (df["vix_spike_20"] & df["is_friday"]).astype(int)
df["yield_shock"] = (df["us10y_pct"].abs() > 0.1).astype(int)

df.dropna(inplace=True)
print(f"Engineered features: {df.shape[1]} columns, {df.shape[0]} rows")

In [ ]:
# Cell 9 — Data summary & label distribution
print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"Date range: {df.index[0]} → {df.index[-1]}")
print(f"Total rows: {len(df)}")
print(f"\nTarget distribution (sp500_direction):")
print(df["sp500_direction"].value_counts())
print(f"\nUp days:   {(df['sp500_direction'] == 1).sum()} ({(df['sp500_direction'] == 1).mean():.1%})")
print(f"Down days: {(df['sp500_direction'] == -1).sum()} ({(df['sp500_direction'] == -1).mean():.1%})")
print(f"\nVIX stats:")
print(df["vix_close"].describe().round(2))

In [ ]:
# Cell 10 — PLOT 1: VIX % change vs SP500 next-day direction
fig, ax = plt.subplots(figsize=(14, 6))
colors = np.where(df["sp500_direction"] == 1, "#00ff88", "#ff4444")
ax.scatter(df["vix_pct"], df["sp500_next_return"], c=colors, alpha=0.4, s=10)
ax.axhline(0, color="white", linewidth=0.5, linestyle="--")
ax.axvline(0, color="white", linewidth=0.5, linestyle="--")
ax.set_xlabel("VIX % Change Today", fontsize=12)
ax.set_ylabel("SP500 Return Next Day (%)", fontsize=12)
ax.set_title("VIX % Change vs SP500 Next-Day Return — Your Edge Validated", fontsize=14)

# Annotate quadrants
ax.text(5, 2, "VIX up → market falls tomorrow\n(YOUR SELL BIAS)", color="#ff4444", fontsize=10, ha="center")
ax.text(-5, 2, "VIX down → market rises tomorrow\n(YOUR BUY BIAS)", color="#00ff88", fontsize=10, ha="center")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 11 — PLOT 2: Correlation heatmap
feature_cols = [
    "vix_pct", "vix_trend", "vix_spike_20", "vix_3day_change",
    "dxy_pct", "dxy_3day", "us10y_pct", "us10y_3day",
    "oil_pct", "oil_3day", "rut_vs_sp500", "rut_leading",
    "overnight_gap", "prev_day_return", "volatility_5d",
    "vix_up_dxy_up", "vix_down_rut_up", "yield_shock",
    "sp500_direction"
]
available = [c for c in feature_cols if c in df.columns]
corr = df[available].corr()

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title("Feature Correlation with SP500 Direction", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 12 — PLOT 3: Win rate by day of week
day_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
day_stats = df.groupby("day_of_week").agg(
    up_days=("sp500_direction", lambda x: (x == 1).sum()),
    total=("sp500_direction", "count"),
).reset_index()
day_stats["win_rate"] = day_stats["up_days"] / day_stats["total"] * 100
day_stats["day_name"] = day_stats["day_of_week"].map(dict(enumerate(day_names)))

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(day_stats["day_name"], day_stats["win_rate"],
              color=["#00ff88" if wr > 52 else "#ff8844" for wr in day_stats["win_rate"]],
              edgecolor="white", linewidth=0.5)
ax.axhline(50, color="white", linewidth=0.5, linestyle="--", alpha=0.5)
for bar, wr in zip(bars, day_stats["win_rate"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{wr:.1f}%", ha="center", va="bottom", fontsize=11, color="white")
ax.set_ylabel("Win Rate (% Up Days)", fontsize=12)
ax.set_title("SP500 Win Rate by Day of Week", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 13 — PLOT 4: Average return by VIX bucket
def vix_bucket(v):
    if v < 15: return "< 15 (Low)"
    elif v < 20: return "15-20 (Normal)"
    elif v < 25: return "20-25 (Elevated)"
    elif v < 30: return "25-30 (High)"
    else: return "> 30 (Extreme)"

df["vix_bucket"] = df["vix_close"].apply(vix_bucket)
bucket_order = ["< 15 (Low)", "15-20 (Normal)", "20-25 (Elevated)", "25-30 (High)", "> 30 (Extreme)"]

bucket_stats = df.groupby("vix_bucket").agg(
    avg_return=("sp500_next_return", "mean"),
    count=("sp500_next_return", "count"),
    win_rate=("sp500_direction", lambda x: (x == 1).mean() * 100),
).reindex(bucket_order)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

colors = ["#00ff88", "#88ff88", "#ffaa44", "#ff8844", "#ff4444"]
ax1.bar(bucket_stats.index, bucket_stats["avg_return"], color=colors, edgecolor="white")
ax1.axhline(0, color="white", linewidth=0.5, linestyle="--")
ax1.set_ylabel("Avg Next-Day Return (%)", fontsize=12)
ax1.set_title("Average Return by VIX Bucket", fontsize=14)
ax1.tick_params(axis='x', rotation=30)

ax2.bar(bucket_stats.index, bucket_stats["win_rate"], color=colors, edgecolor="white")
ax2.axhline(50, color="white", linewidth=0.5, linestyle="--")
ax2.set_ylabel("Win Rate (%)", fontsize=12)
ax2.set_title("Win Rate by VIX Bucket", fontsize=14)
ax2.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 14 — PLOT 5: Individual & combined rule accuracy
# VIX rule: VIX falling → next day UP, VIX rising → next day DOWN
df["vix_rule_pred"] = np.where(df["vix_pct"] < -1.5, 1, np.where(df["vix_pct"] > 1.5, -1, 0))
vix_filtered = df[df["vix_rule_pred"] != 0]
vix_acc = (vix_filtered["vix_rule_pred"] == vix_filtered["sp500_direction"]).mean() * 100

# DXY rule: DXY falling → UP, DXY rising → DOWN
df["dxy_rule_pred"] = np.where(df["dxy_pct"] < -0.2, 1, np.where(df["dxy_pct"] > 0.2, -1, 0))
dxy_filtered = df[df["dxy_rule_pred"] != 0]
dxy_acc = (dxy_filtered["dxy_rule_pred"] == dxy_filtered["sp500_direction"]).mean() * 100

# US10Y rule: yield falling → UP
df["us10y_rule_pred"] = np.where(df["us10y_pct"] < -0.05, 1, np.where(df["us10y_pct"] > 0.05, -1, 0))
us10y_filtered = df[df["us10y_rule_pred"] != 0]
us10y_acc = (us10y_filtered["us10y_rule_pred"] == us10y_filtered["sp500_direction"]).mean() * 100

# RUT rule: RUT leading → UP
df["rut_rule_pred"] = np.where(df["rut_vs_sp500"] > 0.3, 1, np.where(df["rut_vs_sp500"] < -0.3, -1, 0))
rut_filtered = df[df["rut_rule_pred"] != 0]
rut_acc = (rut_filtered["rut_rule_pred"] == rut_filtered["sp500_direction"]).mean() * 100

# Combined VIX + DXY
combined = df[(df["vix_rule_pred"] != 0) & (df["dxy_rule_pred"] != 0) &
              (df["vix_rule_pred"] == df["dxy_rule_pred"])]
combined_acc = (combined["vix_rule_pred"] == combined["sp500_direction"]).mean() * 100 if len(combined) > 0 else 0

rules = ["VIX", "DXY", "US10Y", "RUT", "VIX+DXY"]
accs = [vix_acc, dxy_acc, us10y_acc, rut_acc, combined_acc]
counts = [len(vix_filtered), len(dxy_filtered), len(us10y_filtered), len(rut_filtered), len(combined)]

fig, ax = plt.subplots(figsize=(12, 6))
colors = ["#00ff88" if a > 52 else "#ff8844" for a in accs]
bars = ax.bar(rules, accs, color=colors, edgecolor="white")
ax.axhline(50, color="white", linewidth=0.5, linestyle="--", alpha=0.5, label="Random (50%)")
for bar, acc, cnt in zip(bars, accs, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{acc:.1f}%\n(n={cnt})", ha="center", va="bottom", fontsize=11, color="white")
ax.set_ylabel("Accuracy (%)", fontsize=12)
ax.set_title("Rule Accuracy: Individual & Combined", fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 15 — Final validation printout
print("=" * 50)
print("RULE VALIDATION RESULTS")
print("=" * 50)
print(f"VIX rule accuracy:         {vix_acc:.1f}%  (n={len(vix_filtered)})")
print(f"DXY rule accuracy:         {dxy_acc:.1f}%  (n={len(dxy_filtered)})")
print(f"US10Y rule accuracy:       {us10y_acc:.1f}%  (n={len(us10y_filtered)})")
print(f"RUT rule accuracy:         {rut_acc:.1f}%  (n={len(rut_filtered)})")
print(f"Combined VIX+DXY accuracy: {combined_acc:.1f}%  (n={len(combined)})")
print()

best_day_idx = day_stats["win_rate"].idxmax()
worst_day_idx = day_stats["win_rate"].idxmin()
print(f"Best day to trade:  {day_stats.loc[best_day_idx, 'day_name']} ({day_stats.loc[best_day_idx, 'win_rate']:.1f}%)")
print(f"Worst day to trade: {day_stats.loc[worst_day_idx, 'day_name']} ({day_stats.loc[worst_day_idx, 'win_rate']:.1f}%)")
print()

# Most predictive single feature
target_corr = df[available].corr()["sp500_direction"].drop("sp500_direction").abs().sort_values(ascending=False)
print(f"Most predictive feature: {target_corr.index[0]} (|r| = {target_corr.iloc[0]:.3f})")
print(f"Top 5 features by |correlation|:")
for feat, r in target_corr.head(5).items():
    print(f"  {feat:30s} |r| = {r:.4f}")

In [ ]:
# Cell 16 — Save final features parquet
output_path = os.path.join(DATA_FEAT, "historical_features.parquet")
df.to_parquet(output_path)
print(f"\n✅ Features saved to: {output_path}")
print(f"   Shape: {df.shape}")
print(f"   Columns: {list(df.columns)}")